In [2]:
import torch
import re
import json
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, Mxfp4Config
from peft import PeftModel

# -------------------------
# CONFIG
# -------------------------
BASE_MODEL = "unsloth/gpt-oss-20b"

MODEL_NAME = "top_8"   # change: base / v / v_o / mlp
# LORA_PATHS = {
#     "base": None,
#     "v": "lora_v_proj",
#     "v_o": "lora_v_proj_o_proj",
#     "mlp": "lora_mlp",
# }
LORA_PATHS = {
    "top_4": "lora_v_o_top_4",
    "top_8": "lora_v_o_top_8",
    "top_half": "lora_v_o_top_half",
}

# -------------------------
# LOAD DATA (1 sample)
# -------------------------
df = pd.read_excel("openDDx.xlsx")

# -------------------------
# LOAD MODEL
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

quant_config = Mxfp4Config()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)

if LORA_PATHS[MODEL_NAME] is not None:
    model = PeftModel.from_pretrained(base_model, LORA_PATHS[MODEL_NAME])
else:
    model = base_model

model.eval()

MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GptOssForCausalLM(
      (model): GptOssModel(
        (embed_tokens): Embedding(201088, 2880, padding_idx=200017)
        (layers): ModuleList(
          (0-15): 16 x GptOssDecoderLayer(
            (self_attn): GptOssAttention(
              (q_proj): Linear(in_features=2880, out_features=4096, bias=True)
              (k_proj): Linear(in_features=2880, out_features=512, bias=True)
              (v_proj): Linear(in_features=2880, out_features=512, bias=True)
              (o_proj): Linear(in_features=4096, out_features=2880, bias=True)
            )
            (mlp): GptOssMLP(
              (router): GptOssTopKRouter()
              (experts): GptOssExperts()
            )
            (input_layernorm): GptOssRMSNorm((2880,), eps=1e-05)
            (post_attention_layernorm): GptOssRMSNorm((2880,), eps=1e-05)
          )
          (16-23): 8 x GptOssDecoderLayer(
            (self_attn): GptOssAttention(
              (q

In [13]:
def build_prompt(row):
    return f"""<|system|>
You are a clinical reasoning assistant.

Task:
Read the patient case and determine the most likely diagnosis.

Instructions:
- Think step by step using only the patient information
- Keep reasoning brief (2–4 sentences)

Answer format:
Reasoning: <your reasoning>
Final Answer: <one most likely diagnosis>

<|user|>
PATIENT CASE:
specialty: {row['specialty']}
patient_text: \"\"\"{row['patient_info']}\"\"\"

<|assistant|>
Reasoning:"""

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "<|assistant|>" in text:
        text = text.split("<|assistant|>")[1]

    return text.strip()

In [4]:
def extract_disease_keys(s):
    import json
    import ast

    try:
        # try proper JSON first
        data = json.loads(s)
    except:
        try:
            # fallback for python dict-like strings
            data = ast.literal_eval(s)
        except:
            return None

    if isinstance(data, dict):
        return list(data.keys())
    
    return None

df['diagnosis'] = df['interpretation'].apply(extract_disease_keys)

In [17]:
# -------------------------
# RUN
# -------------------------
row = df.iloc[10]   # change index to test different samples
print(row["diagnosis"])
prompt = build_prompt(row)
output = generate(prompt)

# -------------------------
# PRINT
# -------------------------
print("\n====================")
print("PATIENT CASE")
print("====================\n")
print(row["patient_info"])
print(row["disease_num"])

print("\n====================")
print("MODEL OUTPUT")
print("====================\n")
print(output)

['osteoarthritis', 'rheumatoid arthritis', 'gout', 'pseudogout', 'diabetes-related joint disease', 'hemochromatosis']

PATIENT CASE

A 59-year-old man is evaluated for progressive joint pain. There is swelling and tenderness over the first, second, and third metacarpophalangeal joints of both hands. His hand radiograph is shown. He has had diabetes mellitus for 2 years which is not well controlled with medications. Lab studies show a transferrin saturation of 88% and serum ferritin of 1,200 ng/mL. What best represents the etiology of this patient condition?
6

MODEL OUTPUT

Reasoning: The patient is a 59-year-old man with progressive joint pain and swelling in the first three metacarpophalangeal joints of both hands. His hand radiograph shows changes that are typical of osteoarthritis, but the lab results are unusual. The transferrin saturation is 88%, which is significantly higher than normal, and his serum ferritin is 1,200 ng/mL, which is also elevated. These findings suggest that h

In [18]:
import pandas as pd
import re
from tqdm import tqdm
import torch

# -------------------------
# EXTRACT FUNCTION
# -------------------------
def extract_answer(text):
    if not text:
        return None

    # remove trailing junk
    if "</assistant>" in text:
        text = text.split("</assistant>")[0]

    match = re.search(r'Final Answer:\s*(.*)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip().split("\n")[0]

    return None

# -------------------------
# MAIN LOOP
# -------------------------
all_outputs = []
all_answers = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    try:
        prompt = build_prompt(row)
        output = generate(prompt)

        answer = extract_answer(output)

        all_outputs.append(output)
        all_answers.append(answer)

    except Exception as e:
        print(f"Error at row {i}: {e}")
        all_outputs.append(None)
        all_answers.append(None)

# ------------------------
# SAVE RESULTS
# ------------------------
df["model_output"] = all_outputs
df["final_answer"] = all_answers

df.to_csv("single_diagnosis_results_v_o_top_8_v2.csv", index=False)

print("\nSaved to single_diagnosis_results_v_o_top_8_v2.csv")

100%|██████████| 570/570 [2:44:48<00:00, 17.35s/it]  


Saved to single_diagnosis_results_v_o_top_8_v2.csv
